In [1]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# Load features
image_features = np.load("coco_features/image_features.npy")
caption_features = np.load("coco_features/caption_features.npy")
labels = np.load("coco_features/labels.npy")

# Early fusion
X = np.concatenate([image_features, caption_features], axis=1)  # (5000, 2816)
y = labels  # (5000, 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim=2816, hidden_dim=512, output_dim=80):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()  # multilabel
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 100
batch_size = 64

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))
    total_loss = 0
    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train[idx].to(device), y_train[idx].to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(X_test.to(device))
    test_loss = criterion(preds, y_test.to(device)).item()
print("Final Test Loss:", test_loss)


X shape: (4952, 2816)
y shape: (4952, 80)
Epoch 1/100, Loss: 8.3008
Epoch 2/100, Loss: 4.6877
Epoch 3/100, Loss: 3.9939
Epoch 4/100, Loss: 3.6625
Epoch 5/100, Loss: 3.4338
Epoch 6/100, Loss: 3.2177
Epoch 7/100, Loss: 3.0684
Epoch 8/100, Loss: 2.9067
Epoch 9/100, Loss: 2.8150
Epoch 10/100, Loss: 2.6694
Epoch 11/100, Loss: 2.5311
Epoch 12/100, Loss: 2.4532
Epoch 13/100, Loss: 2.3567
Epoch 14/100, Loss: 2.2525
Epoch 15/100, Loss: 2.1412
Epoch 16/100, Loss: 2.0703
Epoch 17/100, Loss: 1.9533
Epoch 18/100, Loss: 1.8790
Epoch 19/100, Loss: 1.8018
Epoch 20/100, Loss: 1.7069
Epoch 21/100, Loss: 1.6379
Epoch 22/100, Loss: 1.5533
Epoch 23/100, Loss: 1.4753
Epoch 24/100, Loss: 1.4508
Epoch 25/100, Loss: 1.3286
Epoch 26/100, Loss: 1.2505
Epoch 27/100, Loss: 1.1785
Epoch 28/100, Loss: 1.1355
Epoch 29/100, Loss: 1.0650
Epoch 30/100, Loss: 0.9950
Epoch 31/100, Loss: 0.9492
Epoch 32/100, Loss: 0.9039
Epoch 33/100, Loss: 0.8237
Epoch 34/100, Loss: 0.7644
Epoch 35/100, Loss: 0.7140
Epoch 36/100, Loss: 0.

In [3]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()
with torch.no_grad():
    preds = model(X_test.to(device))
    test_loss = criterion(preds, y_test.to(device)).item()
    acc = accuracy_score(y_test, preds.cpu().numpy().round())
print("Final Test Loss:", test_loss)
print("Final Test Accuracy:", acc)

Final Test Loss: 0.1320793628692627
Final Test Accuracy: 0.2946518668012109
